In [ ]:
import numpy as np

x = np.array([[1, 2]])
w = np.array([[0.5, -0.3]])
y = 1
b = 0.1


print(np.round((x @ w.T ) + b))


[[0.]]


In [ ]:
def sigmoid(z):
    return 1 / ( 1 + np.exp(-z) )

def sigmoid_prime(z):
    return sigmoid(z) * ( 1 - sigmoid(z))

def forward(x, w, b):
    z = ( x @ w.T ) + b
    z = np.round(z, decimals=3)
    y_ = sigmoid(z)
    return z, y_

print(f'Forward : {forward(x, w, b)}')

def gradients(x, y, w, b):
    z, y_ = forward(x, w, b)
    n = x.shape[0]
    gw = x.T @ (-(y - y_) * sigmoid_prime(z)) / n
    gb = np.sum(-(y - y_) * sigmoid_prime(z)) / n
    return gw, gb

    """multiplica elemento a elemento as matrizes (n, 1) e multiplica pela de (n, 2) estando transposta, deixando no 
    formato da matriz dos gradientes -> (2, n) @ (n, 1) == (2, 1)"""

print(f'Gradientes {gradients(x, y, w, b)}')

def descida_gradiente(x, y, w, b, alpha=0.1):
    gw, gb = gradients(x, y, w, b)
    gw2 = w - alpha*gw.T
    gb2 = b - alpha*gb
    return gw2, gb2
 
print(f'Novos pesos: {descida_gradiente(x, y,w, b, alpha=0.1)}')

Forward : (array([[0.]]), array([[0.5]]))
Gradientes (array([[-0.125],
       [-0.25 ]]), np.float64(-0.125))
Novos pesos: (array([[ 0.5125, -0.275 ]]), np.float64(0.1125))


In [6]:
"""
X = (n, 2)
Y = (n, 1)
W1 = (3, 2)
b1 = (1, 3)
z2 / a2 = (n, 3)
W2 = (1, 3)
b2 = (1, 1)
z3 / y_ = (n, 1)

delta_3 = (n, 1)
delta_2 = ( (n, 1) @ (1, 3) ) * (n, 3)
          (n, 3) * (n, 3) = (n, 3)

gw1 = (n, 3).T @ (n, 2)
      (3, n) 2 (n, 2) == (3, 2)
      formato exato dos pesos w1!
      pronto pra backforward

gb1 = (n, 3) => (1, 3)

gw2 = (n, 1).T @ (n, 3)
      (1, n) @ (n, 3) == (1, 3)
      formato exato dos pesos w2!
    pronto pra backforward

"""

def forward_2(x, w1, b1, w2, b2):
    z2, a2 = forward(x, w1, b1) 
    z3, y_ = forward(a2, w2, b2)
    return z2, a2, z3, y_

def backward(x, w1, b1, w2, b2, y):
    z2, a2, z3, y_ = forward_2(x, w1, b1, w2, b2)
    n = x.shape[0]

    delta_3 = (y_ - y) * sigmoid_prime(z3)
    delta_2 = (delta_3 @ w2) * sigmoid_prime(z2)

    gw1 = delta_2.T @ x / n
    gb1 = np.sum(delta_2, axis=0, keepdims=True) / n
    gw2 = delta_3.T @ a2 / n
    gb2 = np.sum(delta_3, axis=0, keepdims=True) / n

    return gw1, gb1, gw2, gb2

def descida_gradiente(x, w1, b1, w2, b2, y, alpha):
    gw1, gb1, gw2, gb2 = backward(x, w1, b1, w2, b2, y)

    novo_w1 = w1 - alpha*gw1
    novo_b1 = b1 - alpha*gb1
    novo_w2 = w2 - alpha*gw2
    novo_b2 = b2 - alpha*gb2

    return novo_w1, novo_b1, novo_w2, novo_b2

def loop_treinamento(x, w1, b1, w2, b2, y, alpha, loops):

    
    w1_uso = w1
    b1_uso = b1
    w2_uso = w2
    b2_uso = b2

    for i in range(0, loops+1):
        w1_uso, b1_uso, w2_uso, b2_uso = descida_gradiente(
        x, w1_uso, b1_uso, w2_uso, b2_uso, y, alpha
    )

    return w1_uso, b1, w2_uso, b2_uso



In [ ]:
import numpy as np

X = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])  #(4, 2)
Y = np.array([[0.0], [1.0], [1.0], [0.0]]) #(4, 1)

np.random.seed(42)  
W1 = np.random.randn(3, 2)  # (3, 2)
b1 = np.zeros((1, 3))  # (1, 3)
W2 = np.random.randn(1, 3)  # (1, 3)
b2 = np.zeros((1, 1))  # (1, 1)

alpha = 1.0
loops = 5000

w1_treinado, b1_treinado, w2_treinado, b2_treinado = loop_treinamento(
    X, W1, b1, W2, b2, Y, alpha, loops
)

_, _, _, y_pred = forward_2(
    X, w1_treinado, b1_treinado, w2_treinado, b2_treinado
)

print("Predições após o treinamento:")
for i in range(len(X)):
  print(f"Entrada: {X[i]} | Esperado: {Y[i][0]} | Predição: {y_pred[i][0]:.4f}")

Predições após o treinamento:
Entrada: [0. 0.] | Esperado: 0.0 | Predição: 0.6611
Entrada: [0. 1.] | Esperado: 1.0 | Predição: 0.9609
Entrada: [1. 0.] | Esperado: 1.0 | Predição: 0.9788
Entrada: [1. 1.] | Esperado: 0.0 | Predição: 0.0886
